In [11]:
pip install pandas mysql-connector-python sqlalchemy python-dotenv

In [12]:
import pandas as pd
import mysql.connector
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

In [13]:
# Load .env variables
load_dotenv()

True

In [20]:
# MySQL connection details
username = os.getenv("MYSQL_USER", "root")
password = os.getenv("MYSQL_PASSWORD", "")
host = os.getenv("MYSQL_HOST", "localhost")
database = os.getenv("MYSQL_DB", "olist_ecommerce")

In [21]:
# Step 1: Connect to MySQL (without database first)
conn = mysql.connector.connect(
    host=host,
    user=username,
    password=password
)

cursor = conn.cursor()

In [22]:
# Step 2: Create database if not exists
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {database}")
print("Database created or already exists.")

cursor.close()
conn.close()

Database created or already exists.


In [17]:
# Step 3: Create SQLAlchemy engine (with database now)
engine = create_engine(
    f"mysql+mysqlconnector://{username}:{password}@{host}/{database}"
)

In [18]:
base_path = os.path.join("..", "database")

files = {
    "olist_customers": "olist_customers_dataset.csv",
    "olist_orders": "olist_orders_dataset.csv",
    "olist_order_items": "olist_order_items_dataset.csv",
    "olist_order_payments": "olist_order_payments_dataset.csv",
    "olist_order_reviews": "olist_order_reviews_dataset.csv",
    "olist_products": "olist_products_dataset.csv",
    "olist_sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
    "olist_geolocation": "olist_geolocation_dataset.csv"
}

In [19]:
for table_name, file_name in files.items():

    file_path = os.path.join(base_path, file_name)
    df = pd.read_csv(file_path)

    if table_name == "olist_geolocation":
        chunk_size = 20000
    else:
        chunk_size = None

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=chunk_size,
        method="multi"
    )

    print(f"Loaded table: {table_name} | Rows: {df.shape[0]}")

Loaded table: olist_customers | Rows: 99441
Loaded table: olist_orders | Rows: 99441
Loaded table: olist_order_items | Rows: 112650
Loaded table: olist_order_payments | Rows: 103886
Loaded table: olist_order_reviews | Rows: 99224
Loaded table: olist_products | Rows: 32951
Loaded table: olist_sellers | Rows: 3095
Loaded table: product_category_translation | Rows: 71
Loaded table: olist_geolocation | Rows: 1000163
